In [ ]:
import pandas as pd
from datetime import datetime, timedelta

def generate_strict_leader_plan():
    # 1. LOAD ALL 6 SOURCE FILES (Dynamic lookup for naming variations)
    # Files: Reviewer_Final, Reviewer profile, Reviewer calendar, institution, Inspection Visit, visits requirements
    inst = pd.read_csv('Institution.csv')
    revs = pd.read_csv('Reviewer.csv')
    prof = pd.read_csv('Reviewer profile.csv')
    cal = pd.read_csv('Reviewer calendar.csv')
    past = pd.read_csv('Inspection Visit.csv')
    reqs = pd.read_csv('visits requirements.csv')

    # Standardize column headers
    for df in [inst, revs, prof, cal, past, reqs]:
        df.columns = df.columns.str.strip()

    # Pre-parse calendar dates
    cal['Vacation Start Date'] = pd.to_datetime(cal['Vacation Start Date'])
    cal['Vacation End Date'] = pd.to_datetime(cal['Vacation End Date'])

    req_lookup = reqs.set_index('Visit Type').to_dict('index')
    v_types = list(req_lookup.keys())

    plan = []
    start_anchor = datetime(2026, 9, 1)

    # 2. DYNAMIC PLANNING LOOP
    for i, row in inst.iterrows():
        v_type = v_types[i % len(v_types)]
        rules = req_lookup[v_type]

        num_needed = int(rules['Required Members'])
        duration = int(rules['Duration (Days)'])
        is_lead_needed = str(rules['Leader Required']).strip().lower() == 'yes'
        k_key = 'Required Knowledge ' if 'Required Knowledge ' in rules else 'Required Knowledge'
        needed_k = str(rules[k_key]).strip().lower()

        v_start = start_anchor + timedelta(days=(i * 7))
        v_end = v_start + timedelta(days=duration - 1)

        eligible = []
        fail_reasons = {"COI": 0, "Alumni": 0, "Location": 0, "Expertise": 0, "Vacation": 0}

        # Filtering logic for the specific visit
        for _, r in revs.iterrows():
            rid = r['Reviewer ID']

            # CONSTRAINTS
            if str(r['Conflict Institution ID']) == str(row['Institute ID']):
                fail_reasons["COI"] += 1; continue

            p_match = prof[prof['Reviewer ID'] == rid]
            if not p_match.empty and str(row['Institute Name']).lower() in str(p_match.iloc[0]['University of study']).lower():
                fail_reasons["Alumni"] += 1; continue

            if r['Location'] != row['Location']:
                fail_reasons["Location"] += 1; continue

            if needed_k not in str(r['Knowledge Area']).lower():
                fail_reasons["Expertise"] += 1; continue

            on_vacation = any(not (v_end < vs or v_start > ve) for vs, ve in zip(cal[cal['Reviewer ID']==rid]['Vacation Start Date'], cal[cal['Reviewer ID']==rid]['Vacation End Date']))
            if on_vacation:
                fail_reasons["Vacation"] += 1; continue

            eligible.append({'rid': rid, 'role': str(r['Default Role']).lower(), 'score': r['Performance Score']})

        # 3. TEAM ASSEMBLY (1 LEADER + MEMBERS LOGIC)
        eligible_sorted = sorted(eligible, key=lambda x: x['score'], reverse=True)
        selected = []

        if is_lead_needed:
            # 1. Select EXACTLY ONE Leader
            leaders = [e for e in eligible_sorted if 'leader' in e['role']]
            if leaders:
                selected.append(leaders[0])
                # 2. Fill remaining slots with Members only
                members = [e for e in eligible_sorted if 'member' in e['role']]
                selected.extend(members[:num_needed - 1])
        else:
            # No leader requirement: fill by highest performance score
            selected = eligible_sorted[:num_needed]

        # 4. FINAL STATUS & JUSTIFICATION
        status = "Complete" if len(selected) == num_needed else "Incomplete"

        if status == "Incomplete":
            # Detailed Justification breakdown
            just = f"Found {len(selected)}/{num_needed}. Blocked by: "
            blocks = [f"{v} {k}" for k, v in fail_reasons.items() if v > 0]
            justification = just + ", ".join(blocks)
            # Smart Recommendation
            recommendation = "Reallocate experts from other locations" if fail_reasons['Location'] > 0 else "Reschedule visit dates"
        else:
            justification = "N/A"
            recommendation = "N/A"

        plan.append({
            'Institute ID': row['Institute ID'], 'Institute Name': row['Institute Name'],
            'Location': row['Location'], 'Visit Type': v_type, 'Dates': f"{v_start.date()} to {v_end.date()}",
            'Assigned Reviewers': ", ".join([s['rid'] for s in selected]), 'Status': status,
            'Justification': justification, 'Recommendation': recommendation
        })

    # Save to CSV
    pd.DataFrame(plan).to_csv('Optimized_Annual_Visit_Plan.csv', index=False)
    return "Optimized plan generated with specific role logic."

generate_strict_leader_plan()

'Optimized plan generated with specific role logic.'

In [ ]:
import pandas as pd
from datetime import datetime, timedelta

def generate_strict_leader_plan():
    # 1. LOAD ALL 6 SOURCE FILES
    try:
        inst = pd.read_csv('Institution.csv')
        revs = pd.read_csv('Reviewer.csv')
        prof = pd.read_csv('Reviewer profile.csv')
        cal = pd.read_csv('Reviewer calendar.csv')
        past = pd.read_csv('Inspection Visit.csv')
        reqs = pd.read_csv('visits requirements.csv')
    except FileNotFoundError as e:
        return f"Error: Missing file {e.filename}"

    # Standardize column headers
    for df in [inst, revs, prof, cal, past, reqs]:
        df.columns = df.columns.str.strip()

    # Pre-parse calendar dates
    cal['Vacation Start Date'] = pd.to_datetime(cal['Vacation Start Date'])
    cal['Vacation End Date'] = pd.to_datetime(cal['Vacation End Date'])

    req_lookup = reqs.set_index('Visit Type').to_dict('index')
    v_types = list(req_lookup.keys())

    plan = []
    start_anchor = datetime(2026, 9, 1)

    # 2. DYNAMIC PLANNING LOOP
    for i, row in inst.iterrows():
        # Rotate visit types based on institution index
        v_type = v_types[i % len(v_types)]
        rules = req_lookup[v_type]

        num_needed = int(rules['Required Members'])
        duration = int(rules['Duration (Days)'])
        is_lead_needed = str(rules['Leader Required']).strip().lower() == 'yes'
        k_key = 'Required Knowledge' if 'Required Knowledge' in rules else 'Required Knowledge '
        needed_k = str(rules[k_key]).strip().lower()

        v_start = start_anchor + timedelta(days=(i * 7))
        v_end = v_start + timedelta(days=duration - 1)

        eligible = []
        fail_reasons = {"COI/Alumni": 0, "Location": 0, "Expertise": 0, "Vacation": 0, "PastVisit": 0}

        # 3. FILTERING ENGINE
        for _, r in revs.iterrows():
            rid = r['Reviewer ID']

            # Constraint: Expertise
            if needed_k not in str(r['Knowledge Area']).lower():
                fail_reasons["Expertise"] += 1; continue

            # Constraint: Location
            if r['Location'] != row['Location']:
                fail_reasons["Location"] += 1; continue

            # Constraint: Past Visit (Check string inclusion in history)
            past_at_inst = past[past['Institute ID'] == row['Institute ID']]
            if not past_at_inst.empty and any(str(rid) in str(x) for x in past_at_inst['Assigned Reviewers'].values):
                fail_reasons["PastVisit"] += 1; continue

            # Constraint: COI & Alumni
            p_match = prof[prof['Reviewer ID'] == rid]
            is_alumni = not p_match.empty and str(row['Institute Name']).lower() in str(p_match.iloc[0]['University of study']).lower()
            if str(r['Conflict Institution ID']) == str(row['Institute ID']) or is_alumni:
                fail_reasons["COI/Alumni"] += 1; continue

            # Constraint: Vacation
            on_vacation = any(not (v_end < vs or v_start > ve) for vs, ve in zip(cal[cal['Reviewer ID']==rid]['Vacation Start Date'], cal[cal['Reviewer ID']==rid]['Vacation End Date']))
            if on_vacation:
                fail_reasons["Vacation"] += 1; continue

            eligible.append({'rid': rid, 'role': str(r['Default Role']).lower(), 'score': r['Performance Score']})

        # 4. STRICT TEAM ASSEMBLY
        eligible_sorted = sorted(eligible, key=lambda x: x['score'], reverse=True)
        selected = []

        # === STRICT RULE: Renewal Visits = No Leaders ===
        if v_type == "Renewal Visits":
            members_only = [e for e in eligible_sorted if 'member' in e['role']]
            selected = members_only[:num_needed]

        # === STRICT RULE: Other types = Exactly 1 Leader ===
        elif is_lead_needed:
            leaders = [e for e in eligible_sorted if 'leader' in e['role']]
            members = [e for e in eligible_sorted if 'member' in e['role']]

            if leaders:
                selected.append(leaders[0]) # Take best leader
                selected.extend(members[:num_needed - 1]) # Fill with best members

        else:
            # Fallback for types with no lead requirement defined
            selected = eligible_sorted[:num_needed]

        # 5. STATUS, JUSTIFICATION & RECOMMENDATION
        status = "Complete" if len(selected) == num_needed else "Incomplete"

        if status == "Incomplete":
            # Justification
            just_blocks = [f"{v} {k}" for k, v in fail_reasons.items() if v > 0]
            justification = f"Found {len(selected)}/{num_needed}. Blocked by: " + ", ".join(just_blocks)

            # Recommendation Logic
            if fail_reasons['Location'] > 0 and fail_reasons['Expertise'] == 0:
                recommendation = f"Reallocate experts in {needed_k} from other cities."
            elif fail_reasons['Vacation'] > 0:
                recommendation = "Shift visit date by 1 week to avoid staff leave."
            elif fail_reasons['PastVisit'] > 0 or fail_reasons['COI/Alumni'] > 0:
                recommendation = "Use out-of-city staff to ensure neutrality."
            else:
                recommendation = "Expand reviewer database for this specific expertise."
        else:
            justification = "N/A"
            recommendation = "N/A"

        plan.append({
            'Institute ID': row['Institute ID'],
            'Institute Name': row['Institute Name'],
            'Location': row['Location'],
            'Visit Type': v_type,
            'Dates': f"{v_start.date()} to {v_end.date()}",
            'Assigned Reviewers': ", ".join([s['rid'] for s in selected]),
            'Status': status,
            'Justification': justification,
            'Recommendation': recommendation
        })

    # Save to CSV
    output_file = 'Annual_Reviewer_Selection.csv'
    pd.DataFrame(plan).to_csv(output_file, index=False)
    return f"Success! Plan saved to {output_file}"

# Run the process
print(generate_strict_leader_plan())

Success! Plan saved to Annual_Reviewer_Selection.csv


In [ ]:
import pandas as pd
from datetime import datetime, timedelta

def generate_strict_leader_plan():
    # 1. LOAD ALL 6 SOURCE FILES
    try:
        inst = pd.read_csv('Institution.csv')
        revs = pd.read_csv('Reviewer.csv')
        prof = pd.read_csv('Reviewer profile.csv')
        cal = pd.read_csv('Reviewer calendar.csv')
        past = pd.read_csv('Inspection Visit.csv')
        reqs = pd.read_csv('visits requirements.csv')
    except FileNotFoundError as e:
        return f"Error: Missing file {e.filename}"

    for df in [inst, revs, prof, cal, past, reqs]:
        df.columns = df.columns.str.strip()

    cal['Vacation Start Date'] = pd.to_datetime(cal['Vacation Start Date'])
    cal['Vacation End Date'] = pd.to_datetime(cal['Vacation End Date'])

    req_lookup = reqs.set_index('Visit Type').to_dict('index')
    v_types = list(req_lookup.keys())

    plan = []
    start_anchor = datetime(2026, 9, 1)

    # 2. DYNAMIC PLANNING LOOP
    for i, row in inst.iterrows():
        v_type = v_types[i % len(v_types)]
        rules = req_lookup[v_type]

        num_needed = int(rules['Required Members'])
        duration = int(rules['Duration (Days)'])
        is_lead_needed = str(rules['Leader Required']).strip().lower() == 'yes'
        k_key = 'Required Knowledge' if 'Required Knowledge' in rules else 'Required Knowledge '
        needed_k = str(rules[k_key]).strip().lower()

        v_start = start_anchor + timedelta(days=(i * 7))
        v_end = v_start + timedelta(days=duration - 1)

        eligible = []
        # Track reasons for exclusion
        fail_reasons = {"Expertise": 0, "Location": 0, "PastVisit": 0, "COI/Alumni": 0, "Vacation": 0}

        # 3. FILTERING ENGINE
        for _, r in revs.iterrows():
            rid = r['Reviewer ID']

            # Expertise Check
            if needed_k not in str(r['Knowledge Area']).lower():
                fail_reasons["Expertise"] += 1; continue

            # Location Check
            if r['Location'] != row['Location']:
                fail_reasons["Location"] += 1; continue

            # Past Visit Check
            past_at_inst = past[past['Institute ID'] == row['Institute ID']]
            if not past_at_inst.empty and any(str(rid) in str(x) for x in past_at_inst['Assigned Reviewers'].values):
                fail_reasons["PastVisit"] += 1; continue

            # COI & Alumni Check
            p_match = prof[prof['Reviewer ID'] == rid]
            is_alumni = not p_match.empty and str(row['Institute Name']).lower() in str(p_match.iloc[0]['University of study']).lower()
            if str(r['Conflict Institution ID']) == str(row['Institute ID']) or is_alumni:
                fail_reasons["COI/Alumni"] += 1; continue

            # Vacation Check
            on_vacation = any(not (v_end < vs or v_start > ve) for vs, ve in zip(cal[cal['Reviewer ID']==rid]['Vacation Start Date'], cal[cal['Reviewer ID']==rid]['Vacation End Date']))
            if on_vacation:
                fail_reasons["Vacation"] += 1; continue

            eligible.append({'rid': rid, 'role': str(r['Default Role']).lower(), 'score': r['Performance Score']})

        # 4. STRICT TEAM ASSEMBLY
        eligible_sorted = sorted(eligible, key=lambda x: x['score'], reverse=True)
        selected = []

        if v_type == "Renewal Visits":
            members_only = [e for e in eligible_sorted if 'member' in e['role']]
            selected = members_only[:num_needed]
        elif is_lead_needed:
            leaders = [e for e in eligible_sorted if 'leader' in e['role']]
            members = [e for e in eligible_sorted if 'member' in e['role']]
            if leaders:
                selected.append(leaders[0])
                selected.extend(members[:num_needed - 1])
        else:
            selected = eligible_sorted[:num_needed]

        # 5. EXPLAINABLE JUSTIFICATION & RECOMMENDATION
        status = "Complete" if len(selected) == num_needed else "Incomplete"

        if status == "Incomplete":
            # Build a structured explanation
            explanation = []
            if fail_reasons["Expertise"] > 0:
                explanation.append(f"{fail_reasons['Expertise']} lacked {needed_k} expertise")
            if fail_reasons["Location"] > 0:
                explanation.append(f"{fail_reasons['Location']} were outside {row['Location']}")
            if fail_reasons["PastVisit"] > 0:
                explanation.append(f"{fail_reasons['PastVisit']} had a previous visit record here")
            if fail_reasons["COI/Alumni"] > 0:
                explanation.append(f"{fail_reasons['COI/Alumni']} had Conflict of Interest/Alumni status")
            if fail_reasons["Vacation"] > 0:
                explanation.append(f"{fail_reasons['Vacation']} were on vacation")

            justification = f"Found {len(selected)}/{num_needed}. Issues: " + "; ".join(explanation)

            # Recommendation Logic
            if fail_reasons['Location'] > 0 and (len(selected) < num_needed):
                recommendation = f"Source {needed_k} experts from other cities as local pool is exhausted."
            elif fail_reasons['Vacation'] > 0:
                recommendation = "Adjust visit dates to when key staff return from leave."
            elif fail_reasons['Expertise'] > 0 and len(eligible) == 0:
                recommendation = "Subject Matter Expert (SME) required; current database lacks this specialization."
            else:
                recommendation = "Consider reducing team size or utilizing out-of-city reviewers."
        else:
            justification = "All requirements met successfully."
            recommendation = "N/A"

        plan.append({
            'Institute ID': row['Institute ID'],
            'Institute Name': row['Institute Name'],
            'Location': row['Location'],
            'Visit Type': v_type,
            'Dates': f"{v_start.date()} to {v_end.date()}",
            'Assigned Reviewers': ", ".join([s['rid'] for s in selected]),
            'Status': status,
            'Justification': justification,
            'Recommendation': recommendation
        })

    output_file = 'Annual_MOHESR_Plan_Structured.csv'
    pd.DataFrame(plan).to_csv(output_file, index=False)
    return f"Success! Explainable plan saved to {output_file}"

print(generate_strict_leader_plan())

In [ ]:
import pandas as pd
from datetime import datetime, timedelta

def generate_strict_leader_plan():
    # 1. LOAD ALL 6 SOURCE FILES
    try:
        inst = pd.read_csv('Institution.csv')
        revs = pd.read_csv('Reviewer.csv')
        prof = pd.read_csv('Reviewer profile.csv')
        cal = pd.read_csv('Reviewer calendar.csv')
        past = pd.read_csv('Inspection Visit.csv')
        reqs = pd.read_csv('visits requirements.csv')
    except FileNotFoundError as e:
        return f"Error: Missing file {e.filename}"

    for df in [inst, revs, prof, cal, past, reqs]:
        df.columns = df.columns.str.strip()

    cal['Vacation Start Date'] = pd.to_datetime(cal['Vacation Start Date'])
    cal['Vacation End Date'] = pd.to_datetime(cal['Vacation End Date'])

    req_lookup = reqs.set_index('Visit Type').to_dict('index')
    v_types = list(req_lookup.keys())

    plan = []
    start_anchor = datetime(2026, 9, 1)

    # 2. DYNAMIC PLANNING LOOP
    for i, row in inst.iterrows():
        v_type = v_types[i % len(v_types)]
        rules = req_lookup[v_type]

        num_needed = int(rules['Required Members'])
        duration = int(rules['Duration (Days)'])
        is_lead_needed = str(rules['Leader Required']).strip().lower() == 'yes'
        k_key = 'Required Knowledge' if 'Required Knowledge' in rules else 'Required Knowledge '
        needed_k = str(rules[k_key]).strip().lower()

        v_start = start_anchor + timedelta(days=(i * 7))
        v_end = v_start + timedelta(days=duration - 1)

        eligible = []
        # Track reasons for exclusion
        fail_reasons = {"Expertise": 0, "Location": 0, "PastVisit": 0, "COI/Alumni": 0, "Vacation": 0}

        # 3. FILTERING ENGINE
        for _, r in revs.iterrows():
            rid = r['Reviewer ID']

            # Expertise Check
            if needed_k not in str(r['Knowledge Area']).lower():
                fail_reasons["Expertise"] += 1; continue

            # Location Check
            if r['Location'] != row['Location']:
                fail_reasons["Location"] += 1; continue

            # Past Visit Check
            past_at_inst = past[past['Institute ID'] == row['Institute ID']]
            if not past_at_inst.empty and any(str(rid) in str(x) for x in past_at_inst['Assigned Reviewers'].values):
                fail_reasons["PastVisit"] += 1; continue

            # COI & Alumni Check
            p_match = prof[prof['Reviewer ID'] == rid]
            is_alumni = not p_match.empty and str(row['Institute Name']).lower() in str(p_match.iloc[0]['University of study']).lower()
            if str(r['Conflict Institution ID']) == str(row['Institute ID']) or is_alumni:
                fail_reasons["COI/Alumni"] += 1; continue

            # Vacation Check
            on_vacation = any(not (v_end < vs or v_start > ve) for vs, ve in zip(cal[cal['Reviewer ID']==rid]['Vacation Start Date'], cal[cal['Reviewer ID']==rid]['Vacation End Date']))
            if on_vacation:
                fail_reasons["Vacation"] += 1; continue

            eligible.append({'rid': rid, 'role': str(r['Default Role']).lower(), 'score': r['Performance Score']})

        # 4. STRICT TEAM ASSEMBLY
        eligible_sorted = sorted(eligible, key=lambda x: x['score'], reverse=True)
        selected = []

        if v_type == "Renewal Visits":
            members_only = [e for e in eligible_sorted if 'member' in e['role']]
            selected = members_only[:num_needed]
        elif is_lead_needed:
            leaders = [e for e in eligible_sorted if 'leader' in e['role']]
            members = [e for e in eligible_sorted if 'member' in e['role']]
            if leaders:
                selected.append(leaders[0])
                selected.extend(members[:num_needed - 1])
        else:
            selected = eligible_sorted[:num_needed]

        # 5. EXPLAINABLE JUSTIFICATION & RECOMMENDATION
        status = "Complete" if len(selected) == num_needed else "Incomplete"

        if status == "Incomplete":
            # Build a structured explanation
            explanation = []
            if fail_reasons["Expertise"] > 0:
                explanation.append(f"{fail_reasons['Expertise']} lacked {needed_k} expertise")
            if fail_reasons["Location"] > 0:
                explanation.append(f"{fail_reasons['Location']} were outside {row['Location']}")
            if fail_reasons["PastVisit"] > 0:
                explanation.append(f"{fail_reasons['PastVisit']} had a previous visit record here")
            if fail_reasons["COI/Alumni"] > 0:
                explanation.append(f"{fail_reasons['COI/Alumni']} had Conflict of Interest/Alumni status")
            if fail_reasons["Vacation"] > 0:
                explanation.append(f"{fail_reasons['Vacation']} were on vacation")

            justification = f"Found {len(selected)}/{num_needed}. Issues: " + "; ".join(explanation)

            # Recommendation Logic
            if fail_reasons['Location'] > 0 and (len(selected) < num_needed):
                recommendation = f"Source {needed_k} experts from other cities as local pool is exhausted."
            elif fail_reasons['Vacation'] > 0:
                recommendation = "Adjust visit dates to when key staff return from leave."
            elif fail_reasons['Expertise'] > 0 and len(eligible) == 0:
                recommendation = "Subject Matter Expert (SME) required; current database lacks this specialization."
            else:
                recommendation = "Consider reducing team size or utilizing out-of-city reviewers."
        else:
            justification = "All requirements met successfully."
            recommendation = "N/A"

        plan.append({
            'Institute ID': row['Institute ID'],
            'Institute Name': row['Institute Name'],
            'Location': row['Location'],
            'Visit Type': v_type,
            'Dates': f"{v_start.date()} to {v_end.date()}",
            'Assigned Reviewers': ", ".join([s['rid'] for s in selected]),
            'Status': status,
            'Justification': justification,
            'Recommendation': recommendation
        })

    output_file = 'Annual_MOHESR_Plan.csv'
    pd.DataFrame(plan).to_csv(output_file, index=False)
    return f"Success! Explainable plan saved to {output_file}"

print(generate_strict_leader_plan())

Success! Explainable plan saved to Annual_MOHESR_Plan.csv
